# claude

> Claude Code's models through the Agent SDK or the `claude` CLI - an agent with its own harness, not a completion endpoint.

Your tools go out over Claude Code's own channel where that is allowed - an in-process MCP server,
built per turn, nothing written to disk - and as `<tool_call>` tags in the system prompt where it is
not. An enterprise-managed configuration forbids every dynamic MCP server, including that one, so the
prompt is the channel that always works and the backend falls back to it on the first refusal.
`chat.tool_channel` says which one a chat is on. Either way the harness itself gets no tools
(`claude_builtins=()`), because a model offered a real tool-use API will not punctuate a protocol
described in prose. `ClaudeChat.local` is `False`.


In [ ]:
#| default_exp claude

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, shutil, subprocess
from fastcore.all import Path, store_attr, patch, ifnone
from rishi import core
from rishi.core import *

In [ ]:
#| export
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'truncated', 'hitl_policy', 'extract_fence', 'mk_toolspec', 'ToolCall']

In [ ]:
from fastcore.test import test_eq, test_fail

## The wire

One turn is one `query()` or one `claude -p` process. Claude Code has a real system-prompt channel, so
unlike Cursor the briefing goes there and only the conversation is rendered into the prompt.

The tools are the point, and there are two channels for them. Claude Code declares a caller's tools
to the model as an in-process MCP server - `create_sdk_mcp_server`, no process and no file, but still
an entry in the MCP configuration (`{"type": "sdk", ...}`), which is exactly what an organisation
policy refuses. So it is used where it works and abandoned where it does not: `_fallback` reads the
refusal, sets `_mcp_off`, and every turn after that puts the schemas in the system prompt as tags for
`parse_tool_tags` to read back off the reply. One turn is the whole cost of finding out, and the CLI
path never tries, because there the server would have to be declared through a config file.

The loop moves with the channel. On the MCP one Claude Code calls the tool and keeps going inside a
single `query`, so `approve`, the tool-call budget, `max_steps`, `tool_max_len` and `hist` only apply
if the callback applies them - which is what `run_native` is for. A native turn is governed like a
tags turn and its transcript reads the same afterwards.

Which leaves one thing that has to be true for any of it to work: the harness must have no tools of
its own. A model handed both a real tool-use API and a protocol described in prose uses the API, is
told your tool is not there, and says so instead of emitting a tag - it is not a punctuation problem
and no amount of prompt is going to fix it. `claude_builtins=()` sends `--tools ''`, the harness keeps
nothing, and the tag is the only way left to act. Measured over the two paths and three models, that
is the difference between 1 tool call in 10 and 19 in 19, and it takes the prompt from ~68k tokens to
~9.8k at 41 tools because the harness's own schemas leave with it.

What it must *not* do is claim `--strict-mcp-config` / `strict_mcp_config=True`. That flag is refused
outright where an enterprise configuration exists - "You cannot use --strict-mcp-config when an
enterprise MCP config is present" - so the obvious way to say "only my servers, please" is the one
shape a managed machine rejects. Declare nothing, claim nothing.

In [ ]:
#| export
CLAUDE_BIN = 'claude'   #: the CLI rishi drives; override per chat with `bin=`

# Claude Code's own aliases resolve to the latest build of each; the dated ids work too.
opus5    = 'claude-opus-5'
opus48   = 'claude-opus-4-8'
sonnet5  = 'claude-sonnet-5'
sonnet46 = 'claude-sonnet-4-6'
haiku45  = 'claude-haiku-4-5'
fable5   = 'claude-fable-5'

#: Every id above, for anything that wants to offer the list rather than reach for one of them.
CLAUDE_MODELS = {'opus5': opus5, 'opus48': opus48, 'sonnet5': sonnet5, 'sonnet46': sonnet46,
                 'haiku45': haiku45, 'fable5': fable5}

#: Claude Code's own tools this backend refuses by default. The agent gets its tools from the caller;
#: letting the harness shell out as well is a second, ungoverned way to touch the machine.
CLAUDE_DISALLOWED = ('Bash', 'Write', 'Edit', 'NotebookEdit')

#: Claude Code's own toolset, and why the default is none of it. Your tools reach the model as
#: tags in the system prompt; the harness's reach it as a real tool-use API. Offered both, a model
#: takes the real channel, is told your tool does not exist there, and reports it as broken without
#: ever emitting a tag - 1 call in 10 with the harness's toolset, 19 in 19 without it, across
#: haiku/sonnet/opus and both paths. `()` is `--tools ''`: nothing built in, so a tag is the only
#: way to act. `None` restores the default toolset, and with it the failure.
CLAUDE_BUILTINS = ()

#: Declared on every turn, and deliberately empty. A managed configuration forbids *adding* a
#: dynamic MCP server, so this backend adds none - and it must not reach for `--strict-mcp-config`
#: to say so, because that flag is itself refused when an enterprise config is present ("You cannot
#: use --strict-mcp-config when an enterprise MCP config is present"). Declaring nothing and
#: claiming nothing is the only shape a managed machine accepts.
NO_MCP = '{"mcpServers":{}}'

#: The MCP server rishi's own tools are served under, so a name arrives as `mcp__rishi__<tool>`.
MCP_NAME = 'rishi'

def claude_bin(bin=CLAUDE_BIN):
    "Absolute path to the `claude` binary, or a `FileNotFoundError` that says how to get one."
    if (p := shutil.which(bin)): return p
    raise FileNotFoundError(
        f'{bin!r} is not on $PATH. Install Claude Code (https://claude.com/claude-code) and run '
        f'`{bin} /login`; rishi drives it as a subprocess and never reads your credentials.')

def sdk_available():
    "Is the Claude Agent SDK importable here?"
    try:
        from claude_agent_sdk import query  # noqa: F401
        return True
    except ImportError: return False

def claude_via(via=None):
    "Which path to take: what you named, else the SDK when it is installed, else the CLI."
    if via not in (None, 'sdk', 'cli'): raise ValueError(f"via must be 'sdk', 'cli' or None, not {via!r}")
    if via == 'sdk' and not sdk_available(): raise ImportError(
        'via=\'sdk\' needs the Claude Agent SDK: pip install \'rishi[claude]\'. The CLI path needs '
        'none of it - leave via=None and rishi takes whichever is here.')
    return via or ('sdk' if sdk_available() else 'cli')

def norm_claude_usage(u, model=None):
    "Claude Code's usage block -> rishi's, so a Claude turn adds up with a local one."
    if not u: return {}
    cache = (u.get('cache_read_input_tokens') or 0) + (u.get('cache_creation_input_tokens') or 0)
    pt, ct = (u.get('input_tokens') or 0) + cache, u.get('output_tokens') or 0
    return {'prompt_tokens': pt, 'completion_tokens': ct, 'total_tokens': pt + ct,
            'cached_tokens': u.get('cache_read_input_tokens') or 0, 'model': model}

def norm_claude(d, model=None):
    "A Claude Code result -> a rishi `Resp`, with `<tool_call>` tags read out of the text."
    if d.get('is_error'): raise RuntimeError(f"claude failed: {d.get('result') or d.get('subtype')}")
    text, th = split_think(d.get('result') or '')
    text, tcs = parse_tool_tags(text)
    res = {'role': 'assistant', 'content': text}
    if th: res['channels'] = {'thought': th}
    if tcs: res['tool_calls'] = tcs
    res['usage'] = norm_claude_usage(d.get('usage'), model)
    return Resp(res)

In [ ]:
test_eq(claude_via('cli'), 'cli')
test_fail(lambda: claude_via('rest'), contains='must be')
r = norm_claude({'result': 'ok\n<tool_call>\n{"name": "ls", "arguments": {"path": "."}}\n</tool_call>',
                 'usage': {'input_tokens': 10, 'output_tokens': 5, 'cache_read_input_tokens': 90}}, opus5)
test_eq(resp_text(r), 'ok')
test_eq(r['tool_calls'][0]['function']['name'], 'ls')
test_eq(r['usage']['total_tokens'], 105)          # cache reads are prompt tokens too
test_fail(lambda: norm_claude({'is_error': True, 'result': 'nope'}), contains='nope')

## The chat

In [ ]:
#| export
class ClaudeChat(ToolLoopMixin, Chat):
    "Chat against a Claude Code model - the same `rishi.core.Chat` API, over the Agent SDK or the CLI."
    _runtime = 'claude'
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    mk_content, mk_msg, mk_msgs = staticmethod(mk_oai_content), staticmethod(mk_oai_msg), staticmethod(mk_oai_msgs)
    local = False   #: the binary is local, the model is not

    def __init__(self, model=None, *, runtime=None, model_path=None, sp='', messages=None, tools=None,
                 ctx_limit=None, approve=None, tool_max_len=None, max_steps=10, parallel_tools=False,
                 max_parallel_tools=None, final_prompt=dflt_final_prompt_,
                 permission_mode='auto',    # Claude Code's gate on its *own* tools; yours are rishi's
                 claude_builtins=CLAUDE_BUILTINS,   # the harness's own toolset; () -> none, None -> its default
                 claude_tools=None,         # of what `claude_builtins` leaves, what it may use; None -> all
                 claude_disallowed=CLAUDE_DISALLOWED,   # ...and the ones it may never use
                 workspace=None,            # directory Claude Code works in; None -> the cwd
                 effort=None,               # 'low'/'medium'/'high'/'xhigh'/'max'; None -> the default
                 via=None,                  # 'sdk' or 'cli'; None -> the SDK when it is installed
                 native=None,               # carry tools natively; None -> yes on the SDK path, no on the CLI
                 bin=CLAUDE_BIN, timeout=600, settings=None, cbs=None, default_cbs=True, **opts):
        self.model_id = core.split_runtime(model)[1] or opus5
        self._set_tools(tools)
        store_attr('permission_mode,claude_builtins,claude_tools,claude_disallowed,workspace,effort,bin,timeout,settings,opts')
        self.via = claude_via(via)
        if native and not self.use_sdk: raise ValueError(
            "native=True needs via='sdk': the CLI declares an MCP server through a config file, "
            'which is the thing a managed policy refuses. Leave native=None to take the SDK here.')
        self._native, self._mcp_off = native, False
        self.ctx_limit, self._ctx_tokens = ctx_limit, 0
        self._setup(model=model, sp=sp, messages=messages, tools=tools, approve=approve,
                    tool_max_len=tool_max_len, max_steps=max_steps, parallel_tools=parallel_tools,
                    max_parallel_tools=max_parallel_tools, final_prompt=final_prompt, cbs=cbs,
                    default_cbs=default_cbs)

    @property
    def use_sdk(self):
        "Is this chat going through the Agent SDK rather than the CLI?"
        return self.via == 'sdk'

    @property
    def native(self):
        "Are this chat's tools going out on Claude Code's own channel rather than as tags?"
        return bool(self.toolspecs) and self.use_sdk and self._native is not False and not self._mcp_off

    @property
    def tool_channel(self):
        "`'native'` when the harness carries the schemas, `'tags'` when the system prompt does."
        return 'native' if self.native else 'tags'

    @property
    def token_count(self):
        "the prompt this chat would send next. claude code doesn't give you usage. we estimate"
        return est_tokens(self._prompt()) + est_tokens(self._sp())

    def _sp(self):
        "The briefing, plus the tool schemas when nothing else is carrying them."
        return self.sp if self.native else tag_tools_sp(self.toolspecs, self.sp)

    def _prompt(self):
        "This turn's whole conversation as text. The briefing is not in here - it has its own channel."
        return render_prompt(self.hist)

    def _note_usage(self, r):
        "Remember what the turn cost. This is billing volume, not occupancy -- see `token_count`."
        self._ctx_tokens = (r.get('usage') or {}).get('total_tokens') or self._ctx_tokens
        return r

    def _recreate_conv(self):
        "rishi's history moved - eviction, `reconfigure`. Nothing to invalidate: every turn re-sends it."
        pass

    def close(self):
        "Nothing to release: a turn is a process or a `query`, and neither outlives it."
        pass

## The CLI path

`claude -p` with `--output-format json`. An empty `--mcp-config` and no `--strict-mcp-config` is the
whole enterprise story: no dynamic server is declared, and nothing is claimed that a managed machine
would refuse. `--tools ''` is the other half - the harness offers the model none of its own tools, so
the tags are the only channel it has.

In [ ]:
#| export
@patch
def _cmd(self:ClaudeChat, fmt):
    "The `claude` command line for one turn, minus the prompt."
    cmd = [claude_bin(self.bin), '-p', '--output-format', fmt, '--model', self.model_id,
           '--mcp-config', NO_MCP]   # no `--strict-mcp-config`: see `NO_MCP`
    if (sp := self._sp()): cmd += ['--system-prompt', sp]
    if self.permission_mode: cmd += ['--permission-mode', self.permission_mode]
    if self.effort: cmd += ['--effort', self.effort]
    if self.settings: cmd += ['--settings', str(self.settings)]
    if self.workspace: cmd += ['--add-dir', str(self.workspace)]
    # `['']` rather than nothing: `--tools` is variadic, and the empty name is how it is told
    # to offer no built-in tools at all. An empty allowlist is `claude_builtins=()`, not this.
    if self.claude_builtins is not None: cmd += ['--tools', *(self.claude_builtins or [''])]
    if self.claude_tools: cmd += ['--allowed-tools', *self.claude_tools]
    if self.claude_disallowed: cmd += ['--disallowed-tools', *self.claude_disallowed]
    return cmd

@patch
def _run(self:ClaudeChat, fmt, prompt=None):
    """Run one turn and return the finished process; a non-zero exit is the CLI's message, not a traceback.

    The prompt goes in on stdin rather than as the trailing argument. Two reasons, and the first one
    is a bug this had: `--allowed-tools` and `--disallowed-tools` are variadic, so a trailing
    positional is read as one more tool name and the CLI then reports no prompt at all. The second is
    that a rendered conversation outgrows `ARG_MAX` long before it outgrows the context window.
    """
    r = subprocess.run(self._cmd(fmt), input=ifnone(prompt, self._prompt()), capture_output=True,
                       text=True, timeout=self.timeout, cwd=str(self.workspace) if self.workspace else None)
    if r.returncode != 0: raise RuntimeError(f'claude exited {r.returncode}: {(r.stderr or r.stdout).strip()[:400]}')
    return r

## The SDK path

`claude_agent_sdk.query` is one turn, asynchronous, yielding messages. It is also the only path that
can carry tools natively: `_mcp_server` builds the server from `self.toolspecs` each turn and
`_mcp_names` spells them the way Claude Code namespaces them, `mcp__rishi__<tool>`. With no tools, or
after a refusal, the options carry the same empty MCP configuration `_cmd` does.

In [ ]:
#| export
@patch
def _mcp_server(self:ClaudeChat):
    """rishi's tools as an in-process MCP server: no process, no socket, nothing written to disk.

    Each tool is registered under its bare name and reached as `mcp__rishi__<name>`; the callback
    closes over the bare one because that is what `call_tool` looks up in `ns`.
    """
    from claude_agent_sdk import create_sdk_mcp_server, tool
    def mk(sc):
        async def run(args, _n=sc['name']):
            return {'content': [{'type': 'text', 'text': str(run_native(self, _n, args))}]}
        run.__name__ = sc['name']
        return tool(sc['name'], sc.get('description') or '', sc.get('parameters') or {})(run)
    return create_sdk_mcp_server(MCP_NAME, tools=[mk(t['function']) for t in self.toolspecs])

@patch
def _mcp_names(self:ClaudeChat):
    "What the model may call, spelled the way Claude Code namespaces an MCP server's tools."
    return [f"mcp__{MCP_NAME}__{t['function']['name']}" for t in self.toolspecs]

@patch
def _opts(self:ClaudeChat, sp):
    "Agent SDK options for one turn, with no MCP server for a managed policy to refuse."
    from claude_agent_sdk import ClaudeAgentOptions
    # no `max_turns`: it counts the *harness* turns and raises at the cap, and the loop that matters
    # is rishi's own. With no tools the harness can call, one answer is one turn anyway.
    kw = dict(model=self.model_id, system_prompt=sp or None,
              cwd=str(self.workspace) if self.workspace else None,
              permission_mode=self.permission_mode, settings=self.settings,
              mcp_servers={}, strict_mcp_config=False,   # see `NO_MCP`
              disallowed_tools=list(self.claude_disallowed or ()))
    if self.claude_builtins is not None: kw['tools'] = list(self.claude_builtins)
    if self.claude_tools: kw['allowed_tools'] = list(self.claude_tools)
    # last, and overriding both: an empty `tools` restricts the *built-in* set and leaves an MCP
    # server's tools reachable, so the harness still carries nothing of its own here
    if self.native: kw.update(mcp_servers={MCP_NAME: self._mcp_server()}, allowed_tools=self._mcp_names())
    if self.effort: kw['effort'] = self.effort
    return ClaudeAgentOptions(**{**kw, **self.opts})

@patch
def _sdk_events(self:ClaudeChat, prompt, sp):
    "One `query` as `(kind, value)` pairs: `thought`, `text`, and one final `result` dict."
    from claude_agent_sdk import query, AssistantMessage, ResultMessage, TextBlock, ThinkingBlock
    async def _agen():
        text, res = [], None
        async for m in query(prompt=prompt, options=self._opts(sp)):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ThinkingBlock) and b.thinking: yield 'thought', b.thinking
                    elif isinstance(b, TextBlock) and b.text: text.append(b.text); yield 'text', b.text
            elif isinstance(m, ResultMessage): res = m
        if res is None: raise RuntimeError('the Agent SDK ended without a result')
        # `res.result` is the final answer; the accumulated text is the fallback when it is empty
        yield 'result', {'result': res.result or ''.join(text), 'usage': res.usage,
                         'is_error': res.is_error, 'subtype': res.subtype}
    return sync_iter(_agen)

## The two steps `ToolLoopMixin` drives

In [ ]:
#| export
@patch
def _fallback(self:ClaudeChat, e):
    """A managed policy refused the MCP channel, so move to tags and let the caller try once more.

    Learned from the failure rather than declared up front: a configuration at one of the documented
    paths could be detected, but one somewhere nobody documented causes exactly the same refusal and
    is only ever visible here. Setting `_mcp_off` makes `native` false, so the retry goes out as tags
    and a second refusal cannot match - one turn is the whole cost of finding out.
    """
    if not (self.native and mcp_refused(e)): return False
    self._mcp_off = True
    return True

@patch
def _sdk_once(self:ClaudeChat):
    "One `query` to a finished `Resp`; both the transport and `is_error` can raise the refusal."
    out = next(v for k, v in self._sdk_events(self._prompt(), self._sp()) if k == 'result')
    return norm_claude(out, self.model_id)

@patch
def _model_step(self:ClaudeChat, max_output_tokens=None):
    "One wire call, through whichever path this chat uses."
    if not self.use_sdk: return self._note_usage(norm_claude(json.loads(self._run('json').stdout), self.model_id))
    try: return self._note_usage(self._sdk_once())
    except Exception as e:
        if not self._fallback(e): raise
    return self._note_usage(self._sdk_once())

@patch
def _stream_step(self:ClaudeChat, max_output_tokens=None):
    "The same turn, streamed: thinking on its own channel, and tag calls never rendered as prose."
    split, out, sent = StreamSplit(), None, False
    if self.use_sdk:
        # a refusal arrives before any content, so the retry is only safe while nothing has been
        # yielded - past that the caller has seen half a turn and `_fallback` has to stay out of it
        try:
            for kind, v in self._sdk_events(self._prompt(), self._sp()):
                if kind == 'thought': sent = True; yield {'channels': {'thought': v}}
                elif kind == 'text': sent = True; yield from split.feed(v)
                else: out = v
        except Exception as e:
            if sent or not self._fallback(e): raise
            yield from self._stream_step(max_output_tokens); return
        if (out or {}).get('is_error') and not sent and self._fallback(RuntimeError(out.get('result') or out.get('subtype') or '')):
            yield from self._stream_step(max_output_tokens); return
    else:
        proc = subprocess.Popen(self._cmd('stream-json') + ['--include-partial-messages'],
                                stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                                text=True, cwd=str(self.workspace) if self.workspace else None)
        with proc:
            proc.stdin.write(self._prompt()); proc.stdin.close()   # stdin, not argv - see `_run`
            for line in proc.stdout:
                if not (line := line.strip()): continue
                try: o = json.loads(line)
                except json.JSONDecodeError: continue
                if o.get('type') == 'assistant':
                    for b in (o.get('message') or {}).get('content') or []:
                        if b.get('type') == 'thinking' and (t := b.get('thinking')): yield {'channels': {'thought': t}}
                        elif (t := b.get('text')): yield from split.feed(t)
                elif o.get('type') == 'result': out = o
        if out is None: raise RuntimeError(f'claude ended without a result: {proc.stderr.read()[:400]}')
    yield from split.finish()
    self._step_res = self._note_usage(norm_claude(out, self.model_id))

@patch
def _oneshot(self:ClaudeChat, prompt, sp='', think=None, max_tokens=None):
    """Stateless one-shot text, through whichever path this chat uses.

    `think` and `max_tokens` have no switch on either path - Claude Code decides how much a model
    deliberates and how long it may answer for - so they are accepted and ignored rather than
    quietly changing the meaning of a cheap job.
    """
    if self.use_sdk:
        out = next(v for k, v in self._sdk_events(prompt, sp) if k == 'result')
        return resp_text(norm_claude(out, self.model_id))
    return resp_text(norm_claude(json.loads(self._run('json', prompt).stdout), self.model_id))

## Tests

Nothing here starts a model: what is asserted is the command line and the options - which is where the
enterprise contract lives, and the part that fails silently if it regresses.

In [ ]:
def _fn(query: str) -> str:
    "Search the code."
    return ''

c = ClaudeChat('claude/claude-opus-5', sp='be brief', tools=[_fn], via='cli',
               claude_disallowed=('Bash',), messages=['find the parser'])
test_eq(c.model_id, 'claude-opus-5')          # the `claude/` prefix is stripped, the id is not
test_eq(c.local, False)

cmd = c._cmd('json')
# The enterprise contract: no dynamic server is declared, and `--strict-mcp-config` is *not* claimed.
# A managed machine refuses that flag outright, so asking for it is how this path used to fail.
test_eq(cmd[cmd.index('--mcp-config') + 1], NO_MCP)
test_eq('--strict-mcp-config' in cmd, False)
test_eq(cmd[cmd.index('--model') + 1], 'claude-opus-5')
test_eq(cmd[cmd.index('--disallowed-tools') + 1], 'Bash')
# and the prompt is not on the command line at all: `--disallowed-tools` is variadic and would eat it
test_eq('find the parser' in c._prompt(), True)
test_eq(c._prompt() in cmd, False)

# The harness keeps no tools of its own, so a `<tool_call>` tag is the only way the model can act.
# `--tools` is variadic and the empty name is how that is spelled, so `''` is a real argv element.
test_eq(cmd[cmd.index('--tools') + 1], '')
test_eq('--tools' in ClaudeChat('claude/claude-opus-5', via='cli', claude_builtins=None)._cmd('json'), False)
test_eq(ClaudeChat('claude/claude-opus-5', via='cli', claude_builtins=('Read',))._cmd('json').count('Read'), 1)

# the CLI has no channel of its own, so it is always the tags one
test_eq(c.tool_channel, 'tags')
test_eq(ClaudeChat('claude/claude-opus-5', via='cli').native, False)
test_fail(lambda: ClaudeChat('claude/claude-opus-5', via='cli', native=True), contains="needs via='sdk'")

# ...so the schemas have to be somewhere, and the system prompt is where
sp = cmd[cmd.index('--system-prompt') + 1]
test_eq('be brief' in sp and '"_fn"' in sp and '<tool_call>' in sp, True)
# and the briefing is *not* also in the prompt, which has a channel of its own here
test_eq('be brief' not in c._prompt(), True)

In [ ]:
#| eval: false
# The same contract on the SDK path. `eval: false` only because it needs the SDK installed.
# With no tools there is nothing to carry, so nothing is declared and no policy has anything to refuse.
o = ClaudeChat('claude/claude-opus-5', sp='be brief', via='sdk')._opts('be brief')
test_eq(o.mcp_servers, {})
test_eq(o.strict_mcp_config, False)
test_eq(o.max_turns, None)
test_eq(o.system_prompt, 'be brief')
test_eq(o.tools, [])                              # and none of the harness's own, as on the CLI path

# with tools, Claude Code carries them itself and the tag block leaves the prompt
n = ClaudeChat('claude/claude-opus-5', sp='be brief', tools=[_fn], via='sdk')
test_eq(n.tool_channel, 'native')
test_eq(n._sp(), 'be brief')
test_eq(n._mcp_names(), ['mcp__rishi___fn'])
no = n._opts(n._sp())
test_eq(list(no.mcp_servers), ['rishi'])
test_eq((no.allowed_tools, no.tools, no.strict_mcp_config), (['mcp__rishi___fn'], [], False))

# which is an added MCP server like any other, so a managed policy can refuse it. That is learned
# from the refusal, once, and the schemas move back into the prompt for good.
test_eq(n._fallback(RuntimeError('You cannot use --strict-mcp-config when an enterprise MCP config is present')), True)
test_eq(n.tool_channel, 'tags')
test_eq('<tool_call>' in n._sp(), True)
test_eq(n._fallback(RuntimeError('mcp')), False)   # already off; a second refusal has nothing to do
test_eq(n._opts(n._sp()).mcp_servers, {})

# opting out up front is the same path, without paying a turn to find out
test_eq(ClaudeChat('claude/claude-opus-5', tools=[_fn], via='sdk', native=False).tool_channel, 'tags')
test_eq(ClaudeChat('claude/claude-opus-5', tools=[_fn], via='sdk', native=False)._opts('').mcp_servers, {})

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()